# VoxFlow ASR fine-tuning
This notebook prepares a lightweight Whisper-style fine-tuning workflow for personal voice adaptation. It uses a small synthetic/audio demo dataset first, then you can swap in Common Voice and your own recordings.


In [ ]:
# Install dependencies
# %pip install -q datasets transformers peft accelerate evaluate jiwer librosa soundfile torch

import os
import random
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model

MODEL_NAME = "openai/whisper-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Build a small demo dataset with synthetic speech-like audio.
def make_waveform(text: str, duration=1.0, sr=16000):
    t = np.arange(int(sr * duration)) / sr
    tone = 220.0 + len(text) * 10.0
    wave = 0.3 * np.sin(2 * np.pi * tone * t)
    wave += 0.1 * np.sin(2 * np.pi * 440.0 * t)
    return wave.astype(np.float32)

samples = [
    ("Hello there, this is VoxFlow", "hello there this is voxflow"),
    ("I am testing voice recognition", "i am testing voice recognition"),
    ("Please transcribe this sentence", "please transcribe this sentence"),
    ("Open the project and start listening", "open the project and start listening"),
]

rows = []
for text, target in samples:
    audio = make_waveform(text)
    rows.append({"audio": audio, "text": target})

raw_dataset = Dataset.from_list(rows)

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language="English", task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="English", task="transcribe")


def prepare_example(example):
    audio = example["audio"]
    if isinstance(audio, np.ndarray):
        audio = audio.astype(np.float32)
    encoding = processor(audio=audio, sampling_rate=16000, text=example["text"], return_tensors="pt")
    return {"input_features": encoding.input_features[0], "labels": encoding.labels[0]}

prepared = raw_dataset.map(prepare_example, remove_columns=raw_dataset.column_names)

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.forced_decoder_ids = None
model = model.to(DEVICE)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper_demo_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=1,
    save_steps=50,
    logging_steps=10,
    remove_unused_columns=False,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=prepared,
)

trainer.train()
print("ASR fine-tuning demo completed")

# Later phase: replace the synthetic dataset with Common Voice + your recordings.
